# 🔬 Model Robustness Checks — DSI SquarePoint

Продвинутые проверки модели поверх стандартных метрик.  
Показывают не просто "работает ли модель", а **почему работает и насколько надёжно**.

---

## Содержание

1. [Setup](#1) 
2. [Subperiod Stability](#2) — стабильна ли метрика во времени?
3. [Shuffled Target (Null Hypothesis)](#3) — реальный сигнал или шум?
4. [Feature Ablation](#4) — нет ли single point of failure?
5. [Noise Injection](#5) — переобучилась ли модель на точных значениях?
6. [Hyperparameter Sensitivity](#6) — случайный ли выбор параметров?
7. [🚀 Copy-paste блок для любого датасета](#7)


## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
SEED = 42
np.random.seed(SEED)

# ── Синтетический датасет с временной структурой ─────────────────────
n = 2000
dates = pd.date_range('2020-01-01', periods=n, freq='D')
df = pd.DataFrame({
    'date':    dates,
    'glucose': np.random.normal(100, 20, n),
    'bmi':     np.random.normal(26, 5, n),
    'hba1c':   np.random.normal(5.5, 1.2, n),
    'age':     np.random.randint(20, 80, n).astype(float),
})
# Реальный сигнал: glucose + bmi + шум
df['target'] = (
    0.5 * df['glucose'] +
    0.3 * df['bmi'] +
    0.2 * df['hba1c'] * 10 +
    np.random.normal(0, 5, n)
)

FEATURES = ['glucose', 'bmi', 'hba1c', 'age']
TARGET   = 'target'

# Хронологический сплит (важно для временных данных)
split_idx = int(n * 0.8)
train_df  = df.iloc[:split_idx]
test_df   = df.iloc[split_idx:]

X_train = train_df[FEATURES]
y_train = train_df[TARGET]
X_test  = test_df[FEATURES]
y_test  = test_df[TARGET]

# Обучаем baseline модель
model = lgb.LGBMRegressor(
    n_estimators=300, learning_rate=0.05,
    num_leaves=32, random_state=SEED, verbose=-1
)
model.fit(X_train, y_train)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def spear(y_true, y_pred):
    return spearmanr(y_true, y_pred).statistic

real_rmse = rmse(y_test, model.predict(X_test))
real_rho  = spear(y_test, model.predict(X_test))
print(f'Baseline model: RMSE={real_rmse:.4f}  Spearman ρ={real_rho:.4f}')

---
## 2. Subperiod Stability

### Зачем

Общая метрика на test set может скрывать деградацию сигнала во времени.  
Модель могла "поймать" один период и потом сломаться.

### Интерпретация

```
Метрика по четвертям: [1.7, 1.9, 1.8, 1.8] → стабильно ✓ — реальный сигнал
Метрика по четвертям: [2.5, 2.1, 1.9, 0.1] → деградация ✗ — regime change или устаревание
```

### Что это значит на DSI

- **Стабильно** → модель учит устойчивый паттерн, можно доверять
- **Деградирует** → возможно feature drift, seasonality или model decay
- **Улучшается** → странно, возможно data leakage в поздних периодах


In [ ]:
def subperiod_stability(model, test_df, features, target, n_chunks=4):
    """
    Разбивает тест на n_chunks последовательных кусков по времени.
    Считает RMSE и Spearman на каждом.

    Стабильный сигнал: std/mean < 0.15
    Деградирующий:     сильный тренд вниз по чанкам
    """
    chunks = np.array_split(test_df, n_chunks)
    results = []

    print(f'Subperiod Stability ({n_chunks} chunks):')
    print(f'  {"Chunk":<8} {"N":>6} {"RMSE":>8} {"Spearman":>10}')
    print('  ' + '-' * 36)

    for i, chunk in enumerate(chunks):
        X_chunk = chunk[features]
        y_chunk = chunk[target]
        y_pred  = model.predict(X_chunk)

        chunk_rmse = rmse(y_chunk, y_pred)
        chunk_rho  = spear(y_chunk, y_pred)
        results.append({'chunk': f'Q{i+1}', 'n': len(chunk),
                         'RMSE': chunk_rmse, 'Spearman': chunk_rho})
        print(f'  Q{i+1:<7} {len(chunk):>6} {chunk_rmse:>8.4f} {chunk_rho:>10.4f}')

    res_df = pd.DataFrame(results)
    cv_rmse = res_df['RMSE'].std() / res_df['RMSE'].mean()
    cv_rho  = res_df['Spearman'].std() / res_df['Spearman'].mean()

    print(f'  ' + '-' * 36)
    print(f'  CV(RMSE)     = {cv_rmse:.3f}  '
          f'{"✓ stable" if cv_rmse < 0.15 else "⚠ unstable"}')
    print(f'  CV(Spearman) = {cv_rho:.3f}  '
          f'{"✓ stable" if cv_rho < 0.15 else "⚠ unstable"}')

    # Trend check — деградирует ли сигнал?
    rho_vals = res_df['Spearman'].values
    trend = np.polyfit(range(len(rho_vals)), rho_vals, 1)[0]
    print(f'  Trend (Spearman) = {trend:+.4f}/chunk  '
          f'{"⚠ degrading" if trend < -0.05 else "✓ stable"}')

    # Визуализация
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].bar(res_df['chunk'], res_df['RMSE'], color='steelblue', alpha=0.85)
    axes[0].axhline(res_df['RMSE'].mean(), color='red', linestyle='--',
                    label=f'Mean={res_df["RMSE"].mean():.3f}')
    axes[0].set(title='RMSE by Subperiod', ylabel='RMSE')
    axes[0].legend()

    axes[1].plot(res_df['chunk'], res_df['Spearman'], 'o-',
                 color='steelblue', linewidth=2, markersize=8)
    axes[1].axhline(res_df['Spearman'].mean(), color='red', linestyle='--',
                    label=f'Mean={res_df["Spearman"].mean():.3f}')
    axes[1].set(title='Spearman ρ by Subperiod', ylabel='Spearman ρ')
    axes[1].legend()

    sns.despine()
    plt.suptitle('Subperiod Stability Check\n'
                 'Flat = stable signal | Declining = regime change', y=1.02)
    plt.tight_layout()
    plt.show()
    return res_df


stability_results = subperiod_stability(model, test_df, FEATURES, TARGET, n_chunks=4)

---
## 3. Shuffled Target (Null Hypothesis Test)

### Зачем

Перемешиваем y и обучаем ту же модель. Если модель всё равно показывает хорошую метрику — сигнала нет, модель нашла артефакт в данных.

Это **p-value руками**: строим распределение под H₀ и смотрим насколько реальная метрика выбивается.

### Интерпретация

```
Real RMSE = 5.2,  Shuffled mean = 15.8  → сигнал реальный ✓
Real RMSE = 5.2,  Shuffled mean = 6.1   → почти нет сигнала ✗

Real ρ = 0.82,  Shuffled mean = 0.02  → реальный сигнал ✓  
Real ρ = 0.82,  Shuffled mean = 0.71  → подозрительно, проверь leakage ✗
```

### DSI фраза

> *"Я провёл permutation test: на 50 случайных перестановках таргета средний Spearman = 0.02, реальный = 0.81. Z-score = 18.4 → сигнал статистически значим."*


In [ ]:
def shuffled_target_test(model_class, model_params, X_tr, y_tr, X_te, y_te,
                          n_permutations=50):
    """
    Permutation test под нулевой гипотезой.

    H₀: модель не улавливает реального сигнала
    H₁: модель улавливает реальный сигнал

    Строим распределение метрики при случайном y → p-value руками.
    """
    # Реальная метрика
    real_model = model_class(**model_params)
    real_model.fit(X_tr, y_tr)
    real_rho  = spear(y_te, real_model.predict(X_te))
    real_rmse = rmse(y_te,  real_model.predict(X_te))

    # Метрики на перемешанных таргетах
    null_rhos, null_rmses = [], []
    for i in range(n_permutations):
        y_shuffled = np.random.permutation(y_tr)
        m = model_class(**model_params)
        m.fit(X_tr, y_shuffled)
        null_rhos.append(spear(y_te, m.predict(X_te)))
        null_rmses.append(rmse(y_te, m.predict(X_te)))

    null_rhos  = np.array(null_rhos)
    null_rmses = np.array(null_rmses)

    # Z-score: насколько реальная метрика выбивается из null distribution
    z_rho  = (real_rho  - null_rhos.mean())  / null_rhos.std()
    z_rmse = (null_rmses.mean() - real_rmse) / null_rmses.std()

    # p-value: доля permutations где случайная метрика лучше реальной
    p_value = (null_rhos >= real_rho).mean()

    print(f'Shuffled Target Test (n={n_permutations} permutations):')
    print(f'  Real Spearman   : {real_rho:.4f}')
    print(f'  Null mean ± std : {null_rhos.mean():.4f} ± {null_rhos.std():.4f}')
    print(f'  Z-score         : {z_rho:.2f}  '
          f'{"✓ significant" if abs(z_rho) > 2 else "⚠ not significant"}')
    print(f'  p-value         : {p_value:.4f}  '
          f'{"*** " if p_value < 0.001 else "** " if p_value < 0.01 else "* " if p_value < 0.05 else "n.s."}' )
    print()
    print(f'  Real RMSE       : {real_rmse:.4f}')
    print(f'  Null mean ± std : {null_rmses.mean():.4f} ± {null_rmses.std():.4f}')
    print(f'  Z-score (RMSE)  : {z_rmse:.2f}')

    # Визуализация null distribution
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    for ax, null_vals, real_val, label in [
        (axes[0], null_rhos,  real_rho,  'Spearman ρ'),
        (axes[1], null_rmses, real_rmse, 'RMSE'),
    ]:
        sns.histplot(null_vals, bins=20, kde=True, color='steelblue',
                     edgecolor='white', ax=ax, label='Null (shuffled y)')
        ax.axvline(real_val, color='red', linewidth=2.5, linestyle='--',
                   label=f'Real = {real_val:.3f}')
        ax.axvline(null_vals.mean(), color='grey', linewidth=1,
                   linestyle=':', label=f'Null mean = {null_vals.mean():.3f}')
        ax.set(title=f'Null Distribution: {label}\n'
               f'Real far from null = real signal',
               xlabel=label)
        ax.legend(fontsize=8)

    sns.despine()
    plt.tight_layout()
    plt.show()

    return {'real_rho': real_rho, 'null_mean': null_rhos.mean(),
            'z_score': z_rho, 'p_value': p_value}


perm_results = shuffled_target_test(
    lgb.LGBMRegressor,
    {'n_estimators': 200, 'learning_rate': 0.05,
     'num_leaves': 32, 'random_state': SEED, 'verbose': -1},
    X_train, y_train, X_test, y_test,
    n_permutations=50
)

---
## 4. Feature Ablation

### Зачем

Убираем по одной фиче, переобучаем, смотрим на падение метрики.

- Если одна фича даёт 80% качества → **single point of failure** — рискованно
- Если все фичи примерно равномерно важны → модель **robust**
- Если убрать фичу и метрика улучшилась → фича вносила шум

### Отличие от Feature Importance

Feature Importance (gain) показывает вклад **внутри обученной модели**.  
Ablation показывает **реальное влияние на out-of-sample метрику** — честнее.


In [ ]:
def feature_ablation(model_class, model_params, X_tr, y_tr, X_te, y_te,
                      features):
    """
    Убирает каждую фичу по очереди, переобучает, измеряет падение метрики.

    drop > 0   → фича помогает
    drop < 0   → фича вносила шум (без неё лучше!)
    drop > 50% → single point of failure ⚠
    """
    # Baseline с всеми фичами
    baseline_m = model_class(**model_params)
    baseline_m.fit(X_tr[features], y_tr)
    baseline_rho = spear(y_te, baseline_m.predict(X_te[features]))

    print(f'Feature Ablation (baseline Spearman ρ = {baseline_rho:.4f}):')
    print(f'  {"Feature":<20} {"ρ without":>10} {"Drop":>8} {"Drop%":>8} {"Signal"}')
    print('  ' + '-' * 58)

    ablation_results = []
    for feat in features:
        remaining = [f for f in features if f != feat]
        m = model_class(**model_params)
        m.fit(X_tr[remaining], y_tr)
        rho_without = spear(y_te, m.predict(X_te[remaining]))
        drop     = baseline_rho - rho_without
        drop_pct = drop / baseline_rho * 100

        flag = ''
        if drop_pct > 50:  flag = '⚠ CRITICAL'
        elif drop_pct > 20: flag = '⚠ important'
        elif drop < 0:      flag = '✓ removing helps'
        else:               flag = '✓ ok'

        print(f'  {feat:<20} {rho_without:>10.4f} {drop:>+8.4f} {drop_pct:>7.1f}% {flag}')
        ablation_results.append({'feature': feat, 'rho_without': rho_without,
                                  'drop': drop, 'drop_pct': drop_pct})

    abl_df = pd.DataFrame(ablation_results).sort_values('drop', ascending=False)

    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ['coral' if d > 0.2 * baseline_rho else
              'steelblue' if d > 0 else 'green'
              for d in abl_df['drop']]
    ax.barh(abl_df['feature'], abl_df['drop'], color=colors)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set(xlabel='Drop in Spearman ρ (positive = feature helps)',
           title='Feature Ablation\n'
                 'Red = critical | Blue = helpful | Green = removing helps')
    sns.despine()
    plt.tight_layout()
    plt.show()
    return abl_df


ablation_results = feature_ablation(
    lgb.LGBMRegressor,
    {'n_estimators': 200, 'learning_rate': 0.05,
     'num_leaves': 32, 'random_state': SEED, 'verbose': -1},
    X_train, y_train, X_test, y_test,
    features=FEATURES
)

---
## 5. Noise Injection

### Зачем

Добавляем гауссовский шум к признакам и смотрим насколько деградирует модель.

- **Метрика просела на 5–10%** → модель учит устойчивый паттерн ✓
- **Метрика просела на 50%+** → модель переобучилась на точных значениях ✗

### Почему это важно

В реальных данных всегда есть measurement noise. Если модель хрупка к шуму — она не будет работать в продакшне.


In [ ]:
def noise_injection_test(model_class, model_params, X_tr, y_tr, X_te, y_te,
                          features, noise_levels=None):
    """
    Добавляет гауссовский шум σ = noise_level × std(feature).
    Тестирует несколько уровней шума.

    Robust модель: метрика деградирует плавно и умеренно.
    Fragile модель: резкое падение даже при малом шуме.
    """
    if noise_levels is None:
        noise_levels = [0.0, 0.01, 0.05, 0.10, 0.20, 0.50]

    # Baseline без шума
    baseline_m = model_class(**model_params)
    baseline_m.fit(X_tr[features], y_tr)
    baseline_rho  = spear(y_te, baseline_m.predict(X_te[features]))
    baseline_rmse = rmse(y_te,  baseline_m.predict(X_te[features]))

    print(f'Noise Injection Test (baseline ρ = {baseline_rho:.4f}):')
    print(f'  {"Noise σ":>10} {"ρ":>8} {"RMSE":>8} {"ρ drop%":>9} {"Verdict"}')
    print('  ' + '-' * 52)

    noise_results = []
    for noise_level in noise_levels:
        # Добавляем шум к тренировочным данным
        X_tr_noisy = X_tr[features].copy()
        X_te_noisy = X_te[features].copy()
        for feat in features:
            std = X_tr[feat].std()
            X_tr_noisy[feat] += np.random.randn(len(X_tr)) * std * noise_level
            X_te_noisy[feat] += np.random.randn(len(X_te)) * std * noise_level

        m = model_class(**model_params)
        m.fit(X_tr_noisy, y_tr)
        noisy_rho  = spear(y_te, m.predict(X_te_noisy))
        noisy_rmse = rmse(y_te,  m.predict(X_te_noisy))
        drop_pct   = (baseline_rho - noisy_rho) / baseline_rho * 100

        verdict = '✓ robust' if drop_pct < 10 else \
                  '⚠ moderate' if drop_pct < 30 else '✗ fragile'

        print(f'  {noise_level:>10.0%} {noisy_rho:>8.4f} '
              f'{noisy_rmse:>8.4f} {drop_pct:>8.1f}% {verdict}')
        noise_results.append({'noise_level': noise_level, 'rho': noisy_rho,
                               'rmse': noisy_rmse, 'drop_pct': drop_pct})

    nr_df = pd.DataFrame(noise_results)

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(nr_df['noise_level'] * 100, nr_df['rho'], 'o-',
            color='steelblue', linewidth=2, markersize=7)
    ax.axhline(baseline_rho, color='red', linestyle='--', linewidth=1,
               label=f'Baseline ρ = {baseline_rho:.3f}')
    ax.fill_between(nr_df['noise_level'] * 100,
                    baseline_rho * 0.9, baseline_rho,
                    alpha=0.15, color='green', label='Robust zone (< 10% drop)')
    ax.set(xlabel='Noise level (% of feature std)',
           ylabel='Spearman ρ',
           title='Noise Injection Test\n'
                 'Gradual decline = robust | Sharp drop = fragile/overfit')
    ax.legend()
    sns.despine()
    plt.tight_layout()
    plt.show()
    return nr_df


noise_results = noise_injection_test(
    lgb.LGBMRegressor,
    {'n_estimators': 200, 'learning_rate': 0.05,
     'num_leaves': 32, 'random_state': SEED, 'verbose': -1},
    X_train, y_train, X_test, y_test,
    features=FEATURES
)

---
## 6. Hyperparameter Sensitivity

### Зачем

Если метрика сильно скачет при небольших изменениях гиперпараметров — ты нашёл лучший по удаче, а не по логике.

```
num_leaves ∈ [16, 32, 64, 128]:
  метрика = [0.05, 0.18, 0.81, 0.79] → НЕСТАБИЛЬНО ✗ (скачок на 0.05→0.18)
  метрика = [0.78, 0.81, 0.80, 0.79] → СТАБИЛЬНО ✓
```

### DSI фраза

> *"Я провёл sensitivity analysis. В диапазоне num_leaves [32–128] метрика стабильна в [0.79–0.82]. Выбор num_leaves=64 не случаен — результат устойчив к этому гиперпараметру."*

Это намного сильнее чем "я сделал grid search и взял лучшее".


In [ ]:
def hyperparam_sensitivity(model_class, base_params, X_tr, y_tr, X_te, y_te,
                            param_grid):
    """
    Тестирует каждый гиперпараметр независимо, остальные фиксированы.

    Стабильный: CV(metric) по параметру < 0.10
    Нестабильный: CV(metric) > 0.20 → результат случаен
    """
    print('Hyperparameter Sensitivity Analysis:')
    all_results = {}

    for param_name, param_values in param_grid.items():
        metrics = []
        for val in param_values:
            params = {**base_params, param_name: val}
            m = model_class(**params)
            m.fit(X_tr, y_tr)
            rho = spear(y_te, m.predict(X_te))
            metrics.append(rho)

        metrics = np.array(metrics)
        cv = metrics.std() / metrics.mean() if metrics.mean() != 0 else 0
        verdict = '✓ stable' if cv < 0.10 else \
                  '⚠ moderate' if cv < 0.20 else '✗ sensitive'

        print(f'\n  {param_name}:')
        for val, met in zip(param_values, metrics):
            print(f'    {str(val):>10} → ρ = {met:.4f}')
        print(f'  CV = {cv:.3f}  range = [{metrics.min():.4f}, {metrics.max():.4f}]  {verdict}')
        all_results[param_name] = {'values': param_values, 'metrics': metrics, 'cv': cv}

    # Визуализация
    n_params = len(param_grid)
    fig, axes = plt.subplots(1, n_params, figsize=(5 * n_params, 4))
    if n_params == 1: axes = [axes]

    for ax, (param_name, res) in zip(axes, all_results.items()):
        ax.plot(range(len(res['values'])), res['metrics'],
                'o-', color='steelblue', linewidth=2, markersize=8)
        ax.set_xticks(range(len(res['values'])))
        ax.set_xticklabels([str(v) for v in res['values']], rotation=30)
        ax.axhline(res['metrics'].mean(), color='red', linestyle='--',
                   linewidth=1, label=f'mean={res["metrics"].mean():.3f}')
        ax.fill_between(range(len(res['values'])),
                         res['metrics'].mean() - res['metrics'].std(),
                         res['metrics'].mean() + res['metrics'].std(),
                         alpha=0.15, color='steelblue')
        ax.set(title=f'{param_name}\nCV={res["cv"]:.3f}',
               xlabel=param_name, ylabel='Spearman ρ')
        ax.legend(fontsize=8)
        sns.despine(ax=ax)

    plt.suptitle('Hyperparameter Sensitivity\n'
                 'Flat line = stable choice | Spiky = lucky pick', y=1.02)
    plt.tight_layout()
    plt.show()
    return all_results


sensitivity = hyperparam_sensitivity(
    lgb.LGBMRegressor,
    base_params={'n_estimators': 300, 'learning_rate': 0.05,
                 'num_leaves': 64, 'random_state': SEED, 'verbose': -1},
    X_tr=X_train, y_tr=y_train, X_te=X_test, y_te=y_test,
    param_grid={
        'num_leaves':       [16, 32, 64, 128],
        'min_child_samples': [5, 20, 50, 100],
        'learning_rate':    [0.01, 0.05, 0.1, 0.3],
    }
)

---
## 7. 🚀 Copy-Paste блок для любого датасета

Вставь после основного модельного блока. Замени `model`, `X_train`, `y_train`, `X_test`, `y_test`, `test_df`, `FEATURES`.


In [ ]:
# ═══════════════════════════════════════════════════════════
# ROBUSTNESS CHECKS — вставь после обучения модели
# Требования:
#   model     — обученная модель (LightGBM / XGBoost / Ridge)
#   X_train, y_train, X_test, y_test — numpy arrays или DataFrames
#   test_df   — DataFrame с колонкой 'date' (для subperiod)
#   FEATURES  — список признаков
#   MODEL_CLASS, MODEL_PARAMS — для переобучения в тестах
# ═══════════════════════════════════════════════════════════

print('=' * 60)
print('ROBUSTNESS CHECKS')
print('=' * 60)

# ── 1. Subperiod Stability ─────────────────────────────────
print('\n[1/4] Subperiod Stability...')
_ = subperiod_stability(model, test_df, FEATURES, TARGET, n_chunks=4)

# ── 2. Shuffled Target ────────────────────────────────────
print('\n[2/4] Shuffled Target (Permutation Test)...')
MODEL_CLASS  = lgb.LGBMRegressor   # ← замени на свою модель
MODEL_PARAMS = {                    # ← замени на свои параметры
    'n_estimators': 200, 'learning_rate': 0.05,
    'num_leaves': 32, 'random_state': SEED, 'verbose': -1
}
_ = shuffled_target_test(
    MODEL_CLASS, MODEL_PARAMS,
    X_train, y_train, X_test, y_test,
    n_permutations=50   # 50 достаточно, 100 надёжнее
)

# ── 3. Feature Ablation ───────────────────────────────────
print('\n[3/4] Feature Ablation...')
_ = feature_ablation(
    MODEL_CLASS, MODEL_PARAMS,
    X_train, y_train, X_test, y_test,
    features=FEATURES
)

# ── 4. Hyperparameter Sensitivity ────────────────────────
print('\n[4/4] Hyperparameter Sensitivity...')
_ = hyperparam_sensitivity(
    MODEL_CLASS,
    base_params=MODEL_PARAMS,
    X_tr=X_train, y_tr=y_train,
    X_te=X_test,  y_te=y_test,
    param_grid={
        'num_leaves':       [16, 32, 64, 128],   # ← адаптируй под модель
        'min_child_samples': [5, 20, 50, 100],
    }
)

print('\nAll robustness checks complete.')

---
## Как использовать результаты на DSI

### Что говорить по каждой проверке

| Проверка | Хороший результат | Что сказать |
|---|---|---|
| Subperiod | CV < 0.15, нет тренда вниз | *"Сигнал стабилен по всем четвертям, CV=0.08"* |
| Shuffled target | Z-score > 3, p < 0.01 | *"Permutation test: z=12.4, p<0.001 — сигнал не случаен"* |
| Feature ablation | Нет single point of failure | *"Все фичи вносят умеренный вклад, нет critical dependency"* |
| Hyperparam sensitivity | CV < 0.10 по ключевым params | *"Sensitivity analysis показал стабильность в разумном диапазоне"* |

### Если результаты плохие — тоже говори

Плохой результат который ты сам нашёл и объяснил → **плюс**.  
Плохой результат который нашёл интервьюер → **минус**.

> *"Subperiod analysis показал деградацию в Q4 — возможно seasonal effect или feature drift. Следующий шаг: добавить временные признаки или retrain на скользящем окне."*
